In [ ]:
import pandas as pd
import numpy as np
import re
import nltk # type: ignore
from collections import Counter

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger') 


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\arunk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\arunk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\arunk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\arunk\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     C:\Users\arunk\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [33]:
import string

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression



In [3]:
df = pd.read_csv('imdb_dataset.csv')

In [44]:
def preprocess_text(text,
                    lowercase=False,
                    normalize=False,
                    remove_stopwords=False,
                    perserve_negation=False,
                    remove_punctuation=False,
                    keep_important_punctuation=False,
                    stem=False,
                    lemmatize=False):
    
    if normalize:
        text = re.sub(r'(https?://\S+|www\.\S+)', ' ', text)
        text = re.sub(r'<[^>]+>', ' ', text) # Remove HTML tags

    if lowercase:
        text = text.lower()

    if remove_stopwords:
        stop_words = set(stopwords.words('english'))
        if perserve_negation:
            negation_words = ["not", "no", "never", "n't", "cannot", "can't", "don't", "doesn't", "didn't", "won't", "wouldn't", "shouldn't", "couldn't", "isn't", "aren't", "wasn't", "weren't", "haven't", "hasn't", "hadn't"]
            stop_words = stop_words - set(negation_words)
        text = ' '.join([word for word in text.split() if word not in stop_words])    
    
    text = ' '.join([t for t in text.split() if t.isalpha() or t == "n't"])

    if remove_punctuation:
        if keep_important_punctuation:
            #Keep '!' and '?' as they can carry sentiment information
            punctuation_to_remove = string.punctuation.replace('!', '').replace('?', '')
        else:
            punctuation_to_remove = string.punctuation
        text = text.translate(str.maketrans('', '', punctuation_to_remove))
        
    if stem:
        stemmer = PorterStemmer()
        text = ' '.join([stemmer.stem(word) for word in text.split()])
    if lemmatize:
        lemmatizer = WordNetLemmatizer()
        text = ' '.join([lemmatizer.lemmatize(word) for word in text.split()])

    return text


In [45]:
df['preprocessed_text'] = df['review'].apply(lambda x: preprocess_text(x, normalize=True, lowercase=True))

In [46]:
bow_vectorizer = CountVectorizer(max_features=1000, stop_words='english')

x_bow = bow_vectorizer.fit_transform(df['preprocessed_text'])

In [47]:
tfidf_vectorizer = TfidfVectorizer(
    max_features=1000, 
    stop_words='english',
    ngram_range=(1,2))

x_tfidf = tfidf_vectorizer.fit_transform(df['preprocessed_text'])

In [48]:
x_train, x_test, y_train, y_test = train_test_split(x_bow, df['sentiment'], test_size=0.2, random_state=42)

In [50]:
x_train, x_test, y_train, y_test = train_test_split(x_tfidf, df['sentiment'], test_size=0.2, random_state=42)

In [52]:
model = LogisticRegression(max_iter=1000)

model.fit(x_train, y_train)

model.score(x_test, y_test)

0.8412